# Diabetes Regression Model Comparison

## Project overview
This notebook compares several regression models on the diabetes dataset and analyzes predictive performance.

## Skills shown
- Regression benchmarking
- Pipelines and scaling
- Cross-validation
- Model comparison and interpretation


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [3]:
def eval_reg(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:>12} | MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}")


In [4]:
lin = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
lasso = Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.05, max_iter=5000))])

for name, model in [("Linear", lin), ("Ridge", ridge), ("Lasso", lasso)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    eval_reg(name, y_test, pred)


      Linear | MAE=42.79  RMSE=53.85  R2=0.453
       Ridge | MAE=42.81  RMSE=53.78  R2=0.454
       Lasso | MAE=42.80  RMSE=53.77  R2=0.454


In [5]:
rf = RandomForestRegressor(n_estimators=600, random_state=42)
gbr = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=800, random_state=42)

for name, model in [("RF", rf), ("GBR", gbr)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    eval_reg(name, y_test, pred)


          RF | MAE=44.38  RMSE=54.70  R2=0.435
         GBR | MAE=43.77  RMSE=55.29  R2=0.423


In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_rmse = -cross_val_score(ridge, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
lasso_rmse = -cross_val_score(lasso, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")

print("Ridge CV RMSE mean/std:", ridge_rmse.mean(), ridge_rmse.std())
print("Lasso CV RMSE mean/std:", lasso_rmse.mean(), lasso_rmse.std())


Ridge CV RMSE mean/std: 55.374175368673455 2.3732060814135294
Lasso CV RMSE mean/std: 55.369559245603924 2.373493175915404
